# Intelligent Smart Bin - Machine Learning Pipeline
This notebook covers the 3 mandated ML goals for Stage 3:
1. **Temporal Trend Analysis**: Random Forest Regressor to predict fill rates
2. **Usage Pattern Analysis**: K-Means Clustering on location and usage patterns
3. **Anomaly Detection**: Isolation Forest to detect abnormal spikes or broken sensors.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
import joblib
import os

# Ensure models directory exists
os.makedirs('../models', exist_ok=True)

print("Loading Data...")
df = pd.read_csv('../data/sensor_dataset_clean.csv')
df['received_at_iso'] = pd.to_datetime(df['received_at_iso'])
df['hour'] = df['received_at_iso'].dt.hour
df['day_of_week'] = df['received_at_iso'].dt.dayofweek

LOCATION_ENCODING = {
    'Main building': 0, 
    'New building': 1, 
    'SLIIT Basement Canteen': 2, 
    'SLIIT Main Gate': 3,
    'Auditorium': 4
}
df['location_encoded'] = df['location'].map(LOCATION_ENCODING).fillna(0).astype(int)
df['prev_fill'] = df['fill_percentage'].shift(1).bfill()
df.head()

Loading Data...


,device_id,location,fill_percentage,distance_cm,fill_status,moisture_percentage,adc_value,moisture_status,pir_state,received_at_iso,hour
0,BIN_D04,SLIIT Main Gate,11,89,EMPTY,19,3318,DRY,0,2026-04-15 19:03:20.976000+00:00,19
1,BIN_C03,New building,30,70,EMPTY,17,3394,DRY,0,2026-04-15 19:03:20.976000+00:00,19
2,BIN_B02,Main building,6,94,EMPTY,15,3498,DRY,0,2026-04-15 19:03:20.976000+00:00,19
3,BIN_A01,SLIIT Basement Canteen,5,95,EMPTY,12,3586,DRY,0,2026-04-15 19:03:20.976000+00:00,19
4,BIN_D04,SLIIT Main Gate,9,91,EMPTY,17,3401,DRY,0,2026-04-15 20:03:20.976000+00:00,20


### 1. Training Forecasting Model (Random Forest)

In [2]:
# Using 5 features to accurately predict the next fill state
X_reg = df[['hour', 'day_of_week', 'location_encoded', 'fill_percentage', 'prev_fill']].values
y_reg = df['fill_percentage'].shift(-1).bfill().ffill().values

rf_model = RandomForestRegressor(n_estimators=50, random_state=42)
rf_model.fit(X_reg, y_reg)
joblib.dump(rf_model, '../models/forecasting_model.pkl')
print(f"Forecasting Model Trained (Random Forest). R^2 Score: {rf_model.score(X_reg, y_reg):.4f}")

### 2. Usage Pattern Analysis (K-Means Clustering)

In [3]:
# Grouping behavior patterns based on Fill, Moisture, and Time of Day
X_cluster = df[['fill_percentage', 'moisture_percentage', 'hour']].values

kmeans_model = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_model.fit(X_cluster)
joblib.dump(kmeans_model, '../models/usage_cluster_model.pkl')
print("K-Means Behavior Clustering complete.")

Exception in thread Thread-6 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\Startklar\AppData\Local\Programs\Python\Python312\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "C:\Users\Startklar\AppData\Roaming\Python\Python312\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "c:\Users\Startklar\AppData\Local\Programs\Python\Python312\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\Startklar\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "c:\Users\Startklar\AppData\Local\Programs\Python\Python312\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x81 in

K-Means Behavior Clustering complete.


### 3. Anomaly Detection (Isolation Forest)

In [4]:
# Detecting strange behaviors (e.g. sudden massive data jumps)
X_anomaly = df[['fill_percentage', 'pir_state', 'hour']].values
iso_model = IsolationForest(contamination=0.05, random_state=42)
iso_model.fit(X_anomaly)
joblib.dump(iso_model, '../models/anomaly_model.pkl')
print("Isolation Forest Anomaly model saved.")

print("\nAll 3 ML Models successfully trained and saved into model_training/models/ folder!")

Isolation Forest Anomaly model saved.

All 3 ML Models successfully trained and saved into model_training/models/ folder!
